In [1]:
import sys
from pathlib import Path

# Add project root to path to import config
project_root = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(project_root))

from config import PROJECT_ROOT, ANNOTATIONS_DIR, IMAGES_DIR, YOLO_DATA_DIR
print(f"Project root: {PROJECT_ROOT}")
print(f"Annotations: {ANNOTATIONS_DIR}")
print(f"YOLO output: {YOLO_DATA_DIR}")

NameError: name '__file__' is not defined

In [3]:
os.getcwd()

'/mnt/WorkSpace/Repos/PCB-Defect-detection-using-CNN/notebooks'

In [ ]:
annot_path = str(ANNOTATIONS_DIR)
print(f"Annotations path: {annot_path}")

'/mnt/WorkSpace/Repos/PCB-Defect-detection-using-CNN/notebooks/Data/1/PCB_DATASET/Annotations/'

In [ ]:
yolo_data_labels = YOLO_DATA_DIR / 'labels'
yolo_data_labels.mkdir(parents=True, exist_ok=True)

yolo_data_images = YOLO_DATA_DIR / 'images'
yolo_data_images.mkdir(parents=True, exist_ok=True)

print(f"YOLO labels: {yolo_data_labels}")
print(f"YOLO images: {yolo_data_images}")

In [ ]:
import xml.etree.ElementTree as ET
import os
from pathlib import Path

found = False
class_names = {'missing_hole':0, 'mouse_bite':1, 'open_circuit':2, 'short':3, 'spur':4, 'spurious_copper':5}

for dirpath, dirnames, filenames in os.walk(ANNOTATIONS_DIR):
    for filename in filenames:
        if filename.endswith(".xml"):
            found =True
            tree = ET.parse(os.path.join(dirpath, filename))
            root = tree.getroot()
            
            boxes = []
            labels = []
            box_norms = []

            xml_filename, xml_ext = root.find('filename').text.split('.')
            txt_filename = xml_filename + '.txt'

            for size in root.findall('size'):
                height = int(size.find('height').text)
                width  = int(size.find('width').text)
            
            for obj in root.findall('object'):
                label = obj.find('name').text
                bndbox = obj.find('bndbox')
                xmin = int(bndbox.find('xmin').text)
                ymin = int(bndbox.find('ymin').text)
                xmax = int(bndbox.find('xmax').text)
                ymax = int(bndbox.find('ymax').text)
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(label)

                img_class       = class_names[label]
                x_center_norm   = ((xmax + xmin)/2) / width
                y_center_norm   = ((ymax + ymin)/2) / height
                box_width_norm  = ( xmax - xmin)    / width 
                box_heigth_norm = ( ymax - ymin)    / height 

                box_norms.append([img_class, x_center_norm, y_center_norm, box_width_norm, box_heigth_norm]) 

                with open(yolo_data_labels / txt_filename, 'w') as f:
                    for list1 in box_norms:
                        for item in list1:
                            f.write(f'{str(item)} ')
                        f.write('\n')
                
print(f"Processed annotations. Labels saved to: {yolo_data_labels}")

In [ ]:
import shutil 

sub_folder_names = ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']

src_folder = IMAGES_DIR
dst_folder = YOLO_DATA_DIR / 'images'
dst_folder.mkdir(parents=True, exist_ok=True)

for sub_folder in sub_folder_names:
    src_sub_folder = src_folder / sub_folder
    if src_sub_folder.exists():
        for file in src_sub_folder.glob('*.jpg'):  # or '*.png'
            shutil.copy(file, dst_folder / file.name)
            
print(f"Images copied to: {dst_folder}")
print(f"Total images: {len(list(dst_folder.glob('*.jpg')))}")

PermissionError: [Errno 13] Permission denied: '/data_for_yolo'